# Telco Customer Churn — Exploratory Data Analysis

## What you will find in this notebook

This notebook conducts an exploratory data analysis (EDA) in the context of
telecommunications services, focused on understanding the structure, quality and
behaviour of the available variables.

The analysis is organised by groups of variables — demographic, geographic, financial
and relationship — allowing a more coherent reading from a business perspective. For each
group we run univariate and bivariate analyses against the target variable (churn) to
identify patterns, distributions, possible inconsistencies and preliminary signs of
association.

In addition, the final part of the notebook runs a multivariate analysis with an
exploratory approach, including interpretable models such as decision trees, to capture
interactions between variables and deepen the identification of potential churn drivers.

The goal is not only to describe the data but also to generate analytical hypotheses and
guide the next steps in the development of predictive models.

# 1. Business Understanding

This dataset refers to a fictional scenario of a telecommunications company that
offers phone and internet services to its customers. The main objective is to analyse
**customer churn** — i.e. customer attrition, when a user stops using the company's services.

The database captures multiple dimensions of customer behaviour, including demographic
characteristics, contracted services, billing information and tenure with the company. The
target variable indicates whether the customer cancelled the service in a given period.

From a business standpoint, churn analysis is highly strategic, since retaining customers
tends to be significantly cheaper than acquiring new ones. Understanding the factors that
drive attrition allows more effective retention actions — segmented campaigns, customer
experience improvements and service offer adjustments.

## 1.1. Problem Statement
The company is experiencing a high rate of customer churn, leading to revenue loss.
We aim to identify customers at risk of churn and understand the main drivers behind it.

## 1.2. Objectives
- Understand the distribution of churn across customer segments
- Identify features strongly associated with churn
- Build a classification model to predict churn probability

## 1.3. Success Metrics (Business + Technical)
- **Business**
  - Reduce churn by 10%
  - Increase revenue by $6k–$10k/month
- **Model**
  - Recall >= 0.75
  - Precision >= 0.60
  - ROC-AUC >= 0.80
  - Lift >= 2x in top 20%
- **Analytical**
  - Identify top 3–5 churn drivers
  - Produce actionable segments

# 2. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Math / stats
from scipy import stats

from sklearn.tree import DecisionTreeClassifier

# ydata-profiling for automated profiling reports
from ydata_profiling import ProfileReport

# Params
plt.rcParams['figure.dpi'] = 400
pd.set_option('display.max_columns', None)

## 2.1. Helper functions

The analysis below relies on utility helpers organised under `src/utils`
(`utils.eda`, `utils.plots`, `utils.stats`). Import them if that package is available in
your environment.

In [ ]:
# Project-local utility functions (src/utils)
from utils.eda import freq_table, sanity_check, taxa_churn_categoria
from utils.plots import (
    boxplots_target_binaria,
    grid_cat_freq_event_rate,
    plot_univariate,
)
from utils.stats import (
    AnaliseIV,
    calculate_vif,
    chi2_analysis,
    classify_vif,
    cramers_v,
)

# 3. Loading the Dataset

In [ ]:
# Dataset source: https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

# 4. Data Dictionary (Metadata)

The table below describes the variables in the dataset, including type, category and
business interpretation.

| Variable           | Type        | Category            | Description                                            |
|--------------------|-------------|---------------------|--------------------------------------------------------|
| CustomerID         | Categorical | Identification      | Unique customer identifier                             |
| Count              | Numeric     | Technical           | Constant variable (fixed value = 1) used for counting  |
| Country            | Categorical | Geographic          | Customer country                                       |
| State              | Categorical | Geographic          | Customer state                                         |
| City               | Categorical | Geographic          | Customer city                                          |
| Zip Code           | Numeric     | Geographic          | Postal code                                            |
| Lat Long           | Categorical | Geographic          | Combined geographic coordinates                        |
| Latitude           | Numeric     | Geographic          | Latitude                                               |
| Longitude          | Numeric     | Geographic          | Longitude                                              |
| Gender             | Categorical | Demographic         | Customer gender                                        |
| Senior Citizen     | Categorical | Demographic         | Whether the customer is a senior                       |
| Partner            | Categorical | Demographic         | Whether the customer has a partner                     |
| Dependents         | Categorical | Demographic         | Whether the customer has dependents                    |
| Tenure Months      | Numeric     | Relationship        | Customer tenure (months)                               |
| Phone Service      | Categorical | Services            | Has phone service                                      |
| Multiple Lines     | Categorical | Services            | Has multiple lines                                     |
| Internet Service   | Categorical | Services            | Type of internet service                               |
| Online Security    | Categorical | Services            | Online security service                                |
| Online Backup      | Categorical | Services            | Online backup service                                  |
| Device Protection  | Categorical | Services            | Device protection                                      |
| Tech Support       | Categorical | Services            | Technical support                                      |
| Streaming TV       | Categorical | Services            | TV streaming service                                   |
| Streaming Movies   | Categorical | Services            | Movie streaming service                                |
| Contract           | Categorical | Relationship        | Contract type                                          |
| Paperless Billing  | Categorical | Relationship        | Paperless billing                                      |
| Payment Method     | Categorical | Relationship        | Payment method                                         |
| Monthly Charges    | Numeric     | Financial           | Monthly amount charged                                 |
| Total Charges      | Numeric*    | Financial           | Total amount charged (may require type conversion)     |
| Churn Label        | Categorical | Target              | Churn indicator (Yes/No)                               |
| Churn Value        | Binary      | Target              | Numeric churn indicator (1 = churn)                    |
| Churn Score        | Numeric     | Score               | Predictive churn score (derived)                       |
| CLTV               | Numeric     | Financial/Score     | Estimated customer lifetime value                      |
| Churn Reason       | Categorical | Post-Churn          | Cancellation reason (only for churned customers)       |

---

⚠️ **Important notes:**

- *Total Charges* is stored as a string and should be converted to numeric before analysis.
- *Churn Score*, *CLTV* and *Churn Reason* may introduce **data leakage**, since they are
  derived or known only after the churn event.
- *Churn Label* and *Churn Value* represent the same information (use only one as target).

# 5. Sanity Check

In [ ]:
df.info()

In [ ]:
sanity_check(df)

- The `Churn Reason` column has 73.46% missing values. This is expected, as it stores the
  reason a customer left — only available for churned customers.
- Data inconsistencies were found in `Multiple Lines`, `Online Security`, `Online Backup`,
  `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies`, `Payment Method`.
- There are no duplicate rows — each row represents a unique customer.
- The `Count` column was created for dashboard filter manipulation and can be dropped.
- The `Total Charges` column is stored as object instead of float.

## 5.1. Data Treatment

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [ ]:
df['Total Charges'].info()

# 6. Separating the Variables

## 6.1. Variable Classification

The variables were organised into groups according to their nature and role in the business
context, enabling a structured, interpretation-oriented EDA.

### 🔹 Geographic
- Country, State, City, Zip Code, Latitude, Longitude, Lat Long

### 🔹 Demographic
- Gender, Senior Citizen, Partner, Dependents

### 🔹 Customer Relationship
- Tenure Months, Contract, Paperless Billing, Payment Method

### 🔹 Contracted Services
- Phone Service, Multiple Lines, Internet Service, Online Security, Online Backup,
  Device Protection, Tech Support, Streaming TV, Streaming Movies

### 🔹 Financial
- Monthly Charges, Total Charges

### 🔹 Derived / Score
- Churn Score, CLTV

### 🔹 Churn-related (Post-Event ⚠️)
- Churn Label, Churn Value, Churn Reason

### 🔹 Technical / Identification
- CustomerID, Count

---

⚠️ **Important:** variables such as *Churn Score*, *CLTV* and *Churn Reason* may contain
information derived from or posterior to the churn event, and should be used with caution in
predictive analyses to avoid *data leakage*.

In [ ]:
target = 'Churn Label'
num_vars = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
cat_vars = [col for col in df.columns if df[col].dtype in ['str', 'object', 'category']]

## 6.2. Numeric Variables

In [ ]:
num_vars

In [ ]:
geo_vars = ['Zip Code', 'Latitude', 'Longitude']

In [ ]:
# drop geographic variables not used for now
for col in geo_vars:
    num_vars.remove(col)

# drop the Count column - used only for dashboard filter manipulation
num_vars.remove('Count')

In [ ]:
num_vars

## 6.3. Categorical Variables

In [ ]:
cat_vars

In [ ]:
# columns that will not impact the analysis at this first stage
drop_cols = ['CustomerID', 'Country', 'State', 'Lat Long', 'Churn Label', 'Churn Reason']

for col in drop_cols:
    cat_vars.remove(col)

In [ ]:
cat_vars

# 7. Outlier Analysis

In [ ]:
import math

n_cols = 2
n_rows = math.ceil(len(num_vars) / n_cols)

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(12, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_vars):
    sns.boxplot(x=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'Boxplot of {col}', fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.5)

# remove empty axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# 8. Geographic Variables

In [ ]:
df.columns

In [ ]:
vars_geographic = [
    'Country',
    'State',
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude'
]

In [ ]:
df[vars_geographic].nunique()

In [ ]:
df[['Country', 'State']].value_counts()

- The dataset only covers services in the state of California, USA. This may introduce limitations and biases into the churn prediction model.

## 8.1. Frequency Tables

In [ ]:
# absolute frequency
freq_abs = df['City'].value_counts(dropna=False)

# relative frequency (%)
freq_rel = df['City'].value_counts(normalize=True, dropna=False) * 100

# consolidated DataFrame
city_table = pd.DataFrame({
    'City': freq_abs.index,
    'Absolute Frequency': freq_abs.values,
    'Relative Frequency (%)': freq_rel.values
})

# sort (safety)
city_table = city_table.sort_values(by='Absolute Frequency', ascending=False)

# cumulative frequency (%)
city_table['Cumulative Frequency (%)'] = city_table['Relative Frequency (%)'].cumsum()

# top 10 cities
top10_cities = city_table.head(10).reset_index(drop=True)

# round percentage columns to 2 decimals
top10_cities.style.format({
    'Relative Frequency (%)': '{:.2f}',
    'Cumulative Frequency (%)': '{:.2f}'
})

- The dataset shows a classic pattern of high cardinality + low concentration → low raw
  predictive power.
- `City` should not be used without proper cardinality reduction, encoding or conditional
  churn analysis.

## 8.2. Heat Map — Churn Rate by City

In [ ]:
# ensure correct types
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

# drop nulls
df_geo = df.dropna(subset=['Latitude', 'Longitude', 'Churn Value'])

In [ ]:
df_top_regions = (
    df_geo
    .groupby(['City'])
    .agg(
        total_customers=('Churn Value', 'count'),
        churn_rate=('Churn Value', 'mean')
    )
    .query('total_customers >= 30')
    .sort_values('churn_rate', ascending=False)
    .head(10)
)

df_top_regions

In [ ]:
from folium import Map
from folium.plugins import HeatMap

# base map (mean center)
churn_map = Map(
    location=[df_geo['Latitude'].mean(), df_geo['Longitude'].mean()],
    zoom_start=5
)

# heatmap data
heat_data = [
    [row['Latitude'], row['Longitude'], row['Churn Value']]
    for _, row in df_geo.iterrows()
]

# add heatmap
HeatMap(heat_data, radius=8).add_to(churn_map)

churn_map

- Churn is not concentrated; it is systemic across California's urban areas.
- The problem is not a specific city.

In [ ]:
# =========================
# REGION MAPPING (4 CLASSES)
# =========================

NORCAL_COUNTIES = {
    "Alameda","Contra Costa","Marin","Napa","San Francisco","San Mateo",
    "Santa Clara","Solano","Sonoma","Sacramento","Yolo","Placer","El Dorado",
    "Nevada","Yuba","Sutter","Butte","Shasta","Humboldt","Mendocino","Lake",
    "Del Norte","Trinity","Tehama","Siskiyou","Modoc","Lassen","Plumas","Sierra","Alpine","Amador","Calaveras","Tuolumne"
}

SOCAL_COUNTIES = {
    "Los Angeles","Orange","San Diego","Ventura"
}

CENTRAL_COUNTIES = {
    "Fresno","Kern","Kings","Tulare","Madera",
    "San Joaquin","Stanislaus","Merced",
    "Monterey","San Luis Obispo","Santa Barbara","Santa Cruz"
}

DESERT_COUNTIES = {
    "Riverside","San Bernardino","Imperial","Inyo"
}


def classify_region_from_county(county):
    if county in NORCAL_COUNTIES:
        return "norcal"
    elif county in SOCAL_COUNTIES:
        return "socal"
    elif county in CENTRAL_COUNTIES:
        return "central"
    elif county in DESERT_COUNTIES:
        return "socal"  # desert grouped as socal
    else:
        return "other"

In [ ]:
df['City_enc'] = df['City'].apply(classify_region_from_county)

In [ ]:
df['City_enc'].value_counts()

In [ ]:
df.groupby(['City_enc']).agg(
        total_customers=('Churn Value', 'count'),
        churn_rate=('Churn Value', 'mean')
    )

- Outside SoCal, churn is relatively homogeneous.
- SoCal shows a ~14% relative churn difference → strong evidence of a regional effect.
- `other` dominates the dataset (~85%) and may mask more specific patterns.

# 9. Demographic & Family Variables

In [ ]:
vars_demographic = [
    'Gender',
    'Senior Citizen',
    'Partner',
    'Dependents'
]

In [ ]:
df[vars_demographic].describe(include='object')

In [ ]:
for col in vars_demographic:
    print(f"unique values in {col}: {df[col].unique()}")

In [ ]:
for col in vars_demographic:
    print(f"\n📊 Frequency table for {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_demographic:
    print(f'\n📊 Variable: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_demographic,
    target='Churn Value',
    n_cols=2
)

### Distribution
- Gender: almost perfectly balanced
- Senior: relevant minority class (16%)
- Partner: nearly balanced
- Dependents: moderately imbalanced
#### Dataset
- Mostly non-senior population
- Predominantly without dependents
- Balanced marital status
- Gender irrelevant from a distribution standpoint
#### Predictive potential
- Gender: likely irrelevant
- Senior: strong driver candidate, likely a relevant feature
- Partner: indirect behavioural signal
- Dependents: strong stability indicator, high explanatory power
#### Interactions
- Senior + with/without dependents → risk or retention?
- With/without partner + with/without dependents → more stable/volatile profile?
- SoCal + no dependents → very high churn?

# 10. Service Types

In [ ]:
vars_services = [
    'Phone Service',
    'Multiple Lines',
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies'
]

In [ ]:
df[vars_services].describe(include='object')

In [ ]:
for col in vars_services:
    print(f"unique values in {col}: {df[col].unique()}")

In [ ]:
for col in vars_services:
    print(f"\n📊 Frequency table for {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_services:
    print(f'\n📊 Variable: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_services,
    target='Churn Value',
    n_cols=3
)

### Distribution
- Phone Service: almost everyone has phone service
- Multiple Lines: nearly balanced
- Internet Service: good separation between profiles
- Online Security: strong skew
- Online Backup: slightly more adopted, still a minority
- Device Protection: similar to Backup
- Tech Support: low penetration
- Streaming TV: nearly balanced
- Streaming Movies: nearly balanced
#### Conclusion
- "No internet service" and Internet Service = "No" create collinearity, redundancy and
  possible indirect information leakage
- Most customers do not subscribe to: Online Security, Tech Support, Device Protection and
  Online Backup
- Most customers have no add-on services → low engagement with the company
- Hypothesis: without add-on services (especially Online Security and Tech Support), churn
  probability is higher
- Churn is more tied to the level of service engagement than to demographic factors.

# 11. Financial Variables

## 11.1. Contract & Billing

In [ ]:
vars_contract_payment = [
    'Contract',
    'Paperless Billing',
    'Payment Method'
]

In [ ]:
df[vars_contract_payment].describe(include='object')

In [ ]:
for col in vars_contract_payment:
    print(f"unique values in {col}: {df[col].unique()}")

In [ ]:
for col in vars_contract_payment:
    print(f"\n📊 Frequency table for {col}:\n")
    display(freq_table(df, col))

In [ ]:
for var in vars_contract_payment:
    print(f'\n📊 Variable: {var}')
    display(taxa_churn_categoria(df, var))

In [ ]:
grid_cat_freq_event_rate(
    df,
    cat_vars=vars_contract_payment,
    target='Churn Value',
    n_cols=3
)

### Distribution
- Contract: more than half of customers have no lock-in
- Paperless Billing: most use digital billing
- Payment Method: reasonably balanced, slight lead for Electronic Check
#### Insight
- Contract is the biggest potential churn driver. 55% of customers are month-to-month,
  meaning a low exit barrier
- Payment method is a proxy for financial maturity/engagement. Automatic payments are
  associated with higher convenience, less friction and better retention
- Paperless Billing with Electronic Check may indicate a specific behaviour
#### Conclusion
- The company relies heavily on customers without long-term contracts, generating unstable
  revenue and high turnover

## 11.2. Revenue & Monetization

In [ ]:
vars_revenue = [
    'Monthly Charges',
    'Total Charges',
    'Tenure Months'
]

In [ ]:
df[vars_revenue].describe()

In [ ]:
# univariate subplots
for col in vars_revenue:
    plot_univariate(df, col)

- Monthly Charges: KDE shows two well-defined groups; QQ plot confirms a non-Gaussian
  distribution
- Monthly Charges: monthly price segments customers into product tiers
- Total Charges: strongly right-skewed; QQ plot indicates a heavy-tail distribution
- Total Charges: highly correlated with Tenure, may cause overweight in trees
- Tenure: bimodal; QQ plot shows strong deviation from normality
- Tenure: likely a top-3 model feature
#### Conclusion
- There is a clear segmentation between new and old customers, "basic" and "premium"
- Non-linear relationships with churn
- Highly informative variables, especially Tenure

In [ ]:
boxplots_target_binaria(df, 'Churn Label', vars_revenue, n_cols=1)

- Churn is concentrated in higher monthly charges
- Customers who pay more tend to churn
- Churned customers have low accumulated value — they churn early in the lifecycle
- High outliers in churn (Yes) exist, so late churn happens but is rare
- The lower the tenure, the higher the churn
- Churn happens mainly in the customer's first months

# 12. CLTV — Customer Lifetime Value

In [ ]:
df['CLTV'].describe()

In [ ]:
# ===============================
# Style configuration
# ===============================
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram (distribution)
sns.histplot(data=df, x='CLTV', bins=30, kde=True, ax=axes[0])
axes[0].set_title('CLTV Distribution')
axes[0].set_xlabel('CLTV')
axes[0].set_ylabel('Frequency')

# 2. Boxplot vs Churn
sns.boxplot(data=df, x='Churn Label', y='CLTV', hue='Churn Label', palette="Set2", ax=axes[1])
axes[1].set_title('CLTV vs Churn')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('CLTV')

plt.tight_layout()
plt.show()

- The customer base has relatively homogeneous value (no extreme cause)
- Churned customers have lower CLTV
- CLTV has moderate predictive power given the ~300–500 difference in the mean value — a
  visible but not extreme separation

# 13. Churn Score

In [ ]:
df['Churn Score'].describe()

In [ ]:
sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram (distribution)
sns.histplot(data=df, x='Churn Score', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Churn Score Distribution')
axes[0].set_xlabel('Churn Score')
axes[0].set_ylabel('Frequency')

# 2. Boxplot vs Churn
sns.boxplot(data=df, x='Churn Label', y='Churn Score', hue='Churn Label', palette="Set2", ax=axes[1])
axes[1].set_title('Churn Score vs Churn')
axes[1].set_xlabel('Churn Label')
axes[1].set_ylabel('Churn Score')

plt.tight_layout()
plt.show()

- The score already segments customers into risk levels
- churn = 'Yes' is concentrated in high values, median ~80+
- churn = 'No' is dispersed, includes low and medium values, median ~50
- Churn Score separates the classes very well

# 14. Churn Rate

In [ ]:
df['Churn Label'].value_counts(normalize=True)

In [ ]:
# seaborn style
sns.set_theme(style="whitegrid")

# target counts
target_counts = df[target].value_counts().sort_index()
sizes = target_counts.values

# semantic mapping of the target
label_map = {
    'No': 'No Churn',
    'Yes': 'Churn'
}

plt.figure(figsize=(6, 6))
wedges, _, autotexts = plt.pie(
    sizes,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white'}
)

legend_labels = [
    f'{label_map[label]}: {count} observations'
    for label, count in zip(target_counts.index, sizes)
]

plt.legend(
    wedges,
    legend_labels,
    title='Customer Status',
    loc='center left',
    bbox_to_anchor=(1, 0.5)
)

plt.title('Churn Rate')
plt.tight_layout()
plt.show()

- The dataset is moderately imbalanced, but not critically
- Churn is high — around 1 in 4 customers is leaving

## 14.1. Churn Reason

In [ ]:
# churn reason
churn_reason = df['Churn Reason'].value_counts(normalize=True) * 100
churn_reason

Grouping by reason:
- Competition ~33%
  - Competitor higher speed → 10.1%
  - Competitor more data → 8.7%
  - Competitor better offer → 7.5%
  - Competitor better devices → 7.0%
- Experience ~17.5%
  - Attitude of support → 10.3%
  - Attitude of provider → 7.2%
- Product / Quality ~15.8%
  - Network reliability → 5.5%
  - Product dissatisfaction → 5.5%
  - Service dissatisfaction → 4.8%
- Price / Billing ~10.5%
  - Price too high → 5.2%
  - Extra data charges → 3.0%
  - Long distance charges → 2.3%
- Others
  - Don't know → 8.2%
  - Moved → 2.8%
  - Deceased → irrelevant

**Conclusion**
- Churn is not dominated by price; it is dominated by competition + experience
- ~1/3 of churn is due to direct competition
- Poor support weighs as much as price

# 15. Correlation / Association Analysis

## 15.1. Pair Plot

In [ ]:
df_pair = df[num_vars].copy()
df_pair['Churn'] = df[target]
df_pair.drop('Churn Value', axis=1, inplace=True)
df_pair

In [ ]:
sns.pairplot(df_pair, hue='Churn', kind='kde')

- Tenure is the strongest variable
- New customers have high churn risk; old customers are much more stable
- Monthly Charges churn tends to occur at medium-high values
- High price increases churn probability but is not isolated
- Total Charges is not a strong independent variable (it is a consequence of tenure)
- Churn Score = Yes → concentrated in high scores (~70–100)
- Churn = No → concentrated in low scores (~20–60)
- Churn Score has excellent separation → highly predictive feature
- Churn is more present at lower CLTV
- Non-churn is shifted toward higher values
- Churned customers have lower accumulated value

## 15.2. Correlation Matrix

In [ ]:
fig = plt.figure(figsize=(8,6))
sns.heatmap(df[num_vars].corr(),
            fmt=".2f",
            cmap='RdBu_r',
            vmin=-1, vmax=1,
            annot=True)

- Tenure is the most explanatory real variable
- The longer the tenure, the lower the churn — confirming the pairplot
- Tenure × Total Charges → 0.83 (very high), almost linear relationship
- Monthly Charges × Total Charges → 0.65, also a strong relationship

## 15.3. Multicollinearity Analysis — VIF

In [ ]:
vif_df = calculate_vif(df, num_vars)
vif_df = classify_vif(vif_df)
vif_df

- There is strong linear dependence between features
- An almost deterministic implicit relationship exists: Total Charges = Monthly Charges × Tenure
- Churn Score has a 0.66 correlation with the target and is still collinear with other features
- CLTV depends on Tenure and Charges — a derived variable

## 15.4. Information Value (IV)

In [ ]:
target = 'Churn Value'
df_iv = AnaliseIV(df[cat_vars + [target]], target=target, convention="good_over_bad")
df_iv.get_lista_iv()

- IV > 1.0 almost always indicates a problem; abnormally high — City is possibly overfitting
- Contract, Online Security, Tech Support and Internet Service are strong features but may
  contain redundancy and dependence among themselves
- Online Security, Tech Support and Device Protection are highly correlated with each other
- Dependents, Payment Method, Streaming TV / Movies help considerably with less risk
- Paperless Billing, Partner, Senior Citizen are useful as complements
- Gender, Phone Service and Multiple Lines can be removed without impact

In [ ]:
df_iv.get_bivariada('City')

- City is a statistical artifact caused by low frequency + high cardinality

In [ ]:
df_iv.get_bivariada('Contract')

- 88.5% of churn is in month-to-month contracts
- Long contracts concentrate non-churn
- Negative WOE in 'Month-to-month', high churn risk
- High positive WOE in 'Two year', very low risk
- The classes are well separated

## 15.5. Chi-Square + Cramér's V for categorical variables

In [ ]:
result = chi2_analysis(df, cat_vars, target='Churn Value')

display(result)

- p-value is not very informative — almost everything is significant
- City looks strong but is not reliable
- Contract is a genuinely good variable
- Internet Service, Security and Tech Support are good but correlated with each other
- Payment Method gives good separation
- Dependents, streaming and paperless billing are not strong alone but help together
- Gender, Phone Service and Multiple Lines are irrelevant

# 16. Decision Tree for Feature-Importance Analysis

## 16.1. Preparing the Data

### Dummy Variables

In [ ]:
cat_vars.remove('City')
num_vars.remove('Churn Score')

In [ ]:
dummy_vars = pd.get_dummies(df[cat_vars], drop_first=True, dtype=int)
dummy_vars.head()

### df_final

In [ ]:
df_final = pd.concat([df[num_vars], dummy_vars], axis=1)
df_final.head()

### Splitting X and y

In [ ]:
# features
X = df_final.drop(columns=[target])

# target
y = df_final[target]

## 16.2. Plotting the Decision Tree

In [ ]:
exploratory_tree = DecisionTreeClassifier(
    # max_depth=3,
    min_samples_leaf=50,
    random_state=42
)


exploratory_tree.fit(X, y)

In [ ]:
# visualizing the tree
from sklearn import tree
sns.reset_defaults()
fig = plt.figure(figsize=(25, 10))
feature_names = X.columns.to_list()
tree.plot_tree(exploratory_tree, feature_names=feature_names, filled=True, class_names=['No', 'Yes'], fontsize=15)

plt.title('Churn Case Classification\n using a Decision Tree', fontsize=20)
plt.show()

## 16.3. Feature Importance

In [ ]:
importances = np.asarray(exploratory_tree.feature_importances_).ravel()  # ensure 1D numpy
features = list(X.columns)

df_imp = pd.DataFrame({
    "feature": features,
    "importance": importances
})


df_imp["importance"] = pd.to_numeric(df_imp["importance"], errors="coerce")
n_nulls = df_imp["importance"].isna().sum()
if n_nulls > 0:
    print(f"Warning: {n_nulls} importances became NaN during numeric conversion. Check the model/importances.")


df_imp = df_imp.sort_values(by="importance", ascending=False, kind="quicksort").reset_index(drop=True)
df_imp

# 17. Main Churn Drivers

Tier 1 — (core drivers)
- Tenure Months
- Contract
- Internet Service
- Online Security
- Tech Support

Tier 2 — (relevant)
- Dependents
- Payment Method
- Online Backup
- Device Protection

Tier 3 — (moderate)
- Monthly Charges
- Paperless Billing
- Partner
- Senior Citizen

Tier 4 — (irrelevant)
- Gender
- Phone Service
- Multiple Lines

# 18. Main Conclusions

- The current churn rate is **26.6%**;
- The churn risk function is **decreasing over time** — the longer the tenure, the lower the
  churn risk;
- Churn is price-sensitive (monthly charge) but not linear. High price increases churn.
  However, long-tenured customers tolerate high prices while new customers do not;
- Contract type acts as a churn mitigator. Annual contracts have much lower churn rates than
  monthly contracts;
- Among the main churn reasons, poor experience with the company's support was decisive in
  approximately 20% of cases;
- Competition offering better prices and services accounted for about 35% of churn reasons;
- About 25% of churn reasons relate to the mismatch between expectation and delivery of the
  contracted service;
- The `City` variable has high granularity and must be transformed via feature engineering to
  be included in the model;
- `Churn Score` and `CLTV` must be assessed carefully due to leakage risk;
- `Total Charges`, `Tenure Months`, `Monthly Charges`, `CLTV` and `Churn Score` had high VIF,
  resulting in strong multicollinearity.

# 19. Feature Engineering Suggestions

1. Cluster cities into regions to reduce the incidence of rare classes in a single scenario;
2. Digital engagement cluster (service-engagement scorer);
3. `Tenure_Group` flag to stratify new and old customers — churn is not linear over time;
4. Cost-perception proxy;
5. Ordinal encoding of the Contract variable following a monotonic criterion;
6. Family-stability flag identifying customers with dependents or family plans.

# 20. Data Preparation & Profiling

The sections below reproduce the automated-profiling and dataset-preparation pipeline. They
produce a clean, model-ready frame, generate ydata-profiling reports and build a balanced
dataset and a reduced feature set for baseline modeling.

## 20.1. Data Quality Processing

In [ ]:
# drop non-informative / leakage columns
df_dq_processing = df.drop(columns=['Country', 'State', 'Count', 'Lat Long', 'CustomerID', 'Zip Code'])

# convert 'Total Charges' to float
df_dq_processing['Total Charges'] = pd.to_numeric(df_dq_processing['Total Charges'], errors='coerce')

# normalise column names to snake_case
df_dq_processing.columns = (
    df_dq_processing.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.replace(r"__+", "_", regex=True)
    .str.strip("_")
)

In [ ]:
# persist cleaned frame
df_dq_processing.to_parquet("../data/interim/df_dq.parquet")

## 20.2. Profiling Report — Cleaned Dataset

In [ ]:
df_report = pd.read_parquet("../data/interim/df_dq.parquet")

profile = ProfileReport(df_report, title="EDA Report", explorative=True)
profile.to_file("../reports/profiling/report_v1_dq.html")

## 20.3. Balanced Dataset Creation

In [ ]:
# load cleaned frame
df_bal = pd.read_parquet("../data/interim/df_dq.parquet")

# separate classes
df_0 = df_bal[df_bal['churn_value'] == 0]
df_1 = df_bal[df_bal['churn_value'] == 1]

# minority class size
n = min(len(df_0), len(df_1))

# undersample majority class
df_0_sampled = df_0.sample(n=n, random_state=42)
df_1_sampled = df_1.sample(n=n, random_state=42)

# combine and shuffle
df_balanced = pd.concat([df_0_sampled, df_1_sampled]) \
                .sample(frac=1, random_state=42) \
                .reset_index(drop=True)

df_balanced['churn_value'].value_counts()

In [ ]:
df_balanced.to_parquet("../data/processed/df_balanced.parquet", index=False)

## 20.4. Profiling Report — Balanced Dataset

In [ ]:
df_report_bal = pd.read_parquet("../data/processed/df_balanced.parquet")

profile = ProfileReport(df_report_bal, title="Balanced DF Report", explorative=True)
profile.to_file("../reports/profiling/report_v2_balanced.html")

## 20.5. Feature Selection (correlation < 0.3)

Keep a reduced set of relevant columns for baseline modeling.

In [ ]:
df_sel = pd.read_parquet("../data/processed/df_balanced.parquet")

# relevant columns
columns = [
    "churn_reason",
    "contract",
    "dependents",
    "device_protection",
    "internet_service",
    "monthly_charges",
    "online_backup",
    "online_security",
    "payment_method",
    "tech_support",
    "tenure_months",
    "churn_value"
]

df_selected = df_sel[columns]
df_selected.to_parquet("../data/interim/df_selected.parquet", index=False)
df_selected.head()

# 21. Next Steps

- Fill in the project's ML Canvas;
- Build the MVP (baseline model).